In [71]:
# In this assignment you will explore text message data and create models to predict if a message is spam or not.
import pandas as pd
import numpy as np

spam_data = pd.read_csv(
    r"C:\Users\Neelesh Dixit\OneDrive\Desktop\MLModels\spam.csv",
    encoding="latin1"
)

In [78]:
spam_data.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [76]:
spam_data.rename(columns={
    'v1': 'target',
    'v2': 'text'
}, inplace=True)

In [77]:
spam_data['target'] = np.where(
    spam_data['target'] == 'spam',
    1,
    0
)
# 0- ham
# 1- spam 

In [73]:
spam_data = spam_data.drop(columns= ['Unnamed: 2','Unnamed: 3','Unnamed: 4'])

In [79]:
from sklearn.model_selection import train_test_split

In [80]:
X_train, X_test, y_train, y_test = train_test_split(spam_data['text'], 
                                                    spam_data['target'], 
                                                    random_state=0)

In [29]:
# *This function should return a float, the percent value (i.e. $ratio * 100$).*
def answer_one():
    
    a = spam_data['target'].sum()
    b = spam_data['target'].count()
    c = (a * 100) / b
    return c


In [33]:
answer_one()

np.float64(13.406317300789663)

In [81]:
# Fit the training data `X_train` using a Count Vectorizer with default parameters.
# What is the longest token in the vocabulary?
from sklearn.feature_extraction.text import CountVectorizer

In [35]:
def answer_two():
    vocabulary = CountVectorizer().fit(X_train).vocabulary_
    vocabulary = [x for x in vocabulary.keys()]
    len_vocabulary = [len(x) for x in vocabulary]
    
    return vocabulary[np.argmax(len_vocabulary)]

In [36]:
answer_two()

'com1win150ppmx3age16subscription'

In [82]:
#Next, fit a fit a multinomial Naive Bayes classifier model with smoothing `alpha=0.1`. Find the area under the curve (AUC) score using the transformed test data.
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score

In [83]:
def answer_three():
    vect = CountVectorizer().fit(X_train)

    X_train_vectorized = vect.transform(X_train)

    clf = MultinomialNB(alpha=0.1)

    clf.fit(X_train_vectorized, y_train)

    X_test_vectorized = vect.transform(X_test)

    predicted_probability = clf.predict_proba(X_test_vectorized)[:, 1]

    a = roc_auc_score(y_test, predicted_probability)

    return a

In [84]:
answer_three()

0.991545422134696

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def answer_four():
    tfidf = TfidfVectorizer().fit(X_train)
    X_train_tf = tfidf.transform(X_train)

    feature_names = np.array(tfidf.get_feature_names_out())

    max_tf_idfs = X_train_tf.max(0).toarray()[0]

    sorted_tf_idxs = max_tf_idfs.argsort()

    sorted_tf_idfs = max_tf_idfs[sorted_tf_idxs]

    smallest_tf_idfs = pd.Series(
        sorted_tf_idfs[:20],
        index=feature_names[sorted_tf_idxs[:20]]
    )

    largest_tf_idfs = pd.Series(
        sorted_tf_idfs[-20:][::-1],
        index=feature_names[sorted_tf_idxs[-20:][::-1]]
    )

    return (smallest_tf_idfs, largest_tf_idfs)

In [ ]:
answer_four()

In [46]:
def answer_five():
    tfidf = TfidfVectorizer(min_df = 3).fit(X_train)
    X_train_vc = tfidf.transform(X_train)
    clf = MultinomialNB(alpha = 0.1)
    clf.fit(X_train_vc, y_train)
    X_test_vectorized = tfidf.transform(X_test)
    predicted_labels = clf.predict(X_test_vectorized)
    a = roc_auc_score(y_test, predicted_labels)
    return a


In [47]:
answer_five()

0.9416243654822335

In [48]:
def answer_six():
    len_spam = [len(x) for x in spam_data.loc[spam_data['target']==1, 'text']]
    len_not_spam = [len(x) for x in spam_data.loc[spam_data['target'] != 1, 'text']]
    return (np.mean(len_not_spam), np.mean(len_spam))


In [51]:
print(answer_six())

(np.float64(71.02362694300518), np.float64(138.8661311914324))


In [52]:
def add_feature(X, feature_to_add):
    """
    Returns sparse feature matrix with added feature.
    feature_to_add can also be a list of features.
    """
    from scipy.sparse import csr_matrix, hstack
    return hstack([X, csr_matrix(feature_to_add).T], 'csr')

In [53]:
from sklearn.svm import SVC

def answer_seven():
    tfidf = TfidfVectorizer(min_df = 5).fit(X_train)
    X_train_vc = tfidf.transform(X_train)
    X_test_vc = tfidf.transform(X_test)
    clf = SVC(C = 10000)
    len_train = [len(x) for x in X_train]
    len_test = [len(x) for x in X_test]
    X_train_tf = add_feature(X_train_vc, len_train)
    X_test_tf = add_feature(X_test_vc, len_test)
    clf.fit(X_train_tf, y_train)
    predicted = clf.predict(X_test_tf)
    a = roc_auc_score(y_test, predicted)
    return a


In [55]:
answer_seven()

0.9661689557407943

In [56]:
def answer_eight():
    dig_spam = [sum(char.isnumeric() for char in x) for x in spam_data.loc[spam_data['target']==1,'text']]
    dig_not_spam = [sum(char.isnumeric() for char in x) for x in spam_data.loc[spam_data['target']==0,'text']]
    
    return (np.mean(dig_not_spam), np.mean(dig_spam))


In [57]:
answer_eight()

(np.float64(0.2992746113989637), np.float64(15.76037483266399))

In [58]:
from sklearn.linear_model import LogisticRegression

def answer_nine():
    tfidf = TfidfVectorizer(min_df = 5, ngram_range = (1,3)).fit(X_train)
    X_train_vc = tfidf.transform(X_train)
    X_test_vc = tfidf.transform(X_test)
    len_train = [len(x) for x in X_train]
    X_train_tf = add_feature(X_train_vc, len_train)
    dig_train = [sum(char.isnumeric() for char in x) for x in X_train]
    X_train_vc = add_feature(X_train_tf, dig_train)
    len_test = [len(x) for x in X_test]
    X_test_tf = add_feature(X_test_vc, len_test)
    dig_test = [sum(char.isnumeric() for char in x) for x in X_test]
    X_test_vc = add_feature(X_test_tf, dig_test)
    model = LogisticRegression(C = 100)
    model.fit(X_train_vc, y_train)
    predicted = model.predict(X_test_vc)
    a = roc_auc_score(y_test, predicted)
    return a

In [59]:
answer_nine()

0.9733651087380949

In [60]:
def answer_ten():
    len1 = np.mean(spam_data.loc[spam_data['target']==0,'text'].str.count('\W'))
    len2 = np.mean(spam_data.loc[spam_data['target']==1,'text'].str.count('\W'))
    return (len1, len2)

<>:2: SyntaxWarning: invalid escape sequence '\W'
<>:3: SyntaxWarning: invalid escape sequence '\W'
<>:2: SyntaxWarning: invalid escape sequence '\W'
<>:3: SyntaxWarning: invalid escape sequence '\W'
C:\Users\Neelesh Dixit\AppData\Local\Temp\ipykernel_29436\1027462605.py:2: SyntaxWarning: invalid escape sequence '\W'
  len1 = np.mean(spam_data.loc[spam_data['target']==0,'text'].str.count('\W'))
C:\Users\Neelesh Dixit\AppData\Local\Temp\ipykernel_29436\1027462605.py:3: SyntaxWarning: invalid escape sequence '\W'
  len2 = np.mean(spam_data.loc[spam_data['target']==1,'text'].str.count('\W'))


In [61]:
answer_ten()

(np.float64(17.29181347150259), np.float64(29.041499330655956))

In [69]:
def answer_eleven():
    len_train = [len(x) for x in X_train]
    len_test = [len(x) for x in X_test]

    dig_train = [sum(char.isnumeric() for char in x) for x in X_train]
    dig_test = [sum(char.isnumeric() for char in x) for x in X_test]

    nan_train = X_train.str.count(r'\W')
    nan_test = X_test.str.count(r'\W')

    cv = CountVectorizer(
        min_df=5,
        ngram_range=(2,5),
        analyzer='char_wb'
    ).fit(X_train)

    X_train_cv = cv.transform(X_train)
    X_test_cv = cv.transform(X_test)

    X_train_cv = add_feature(
        X_train_cv,
        [len_train, dig_train, nan_train]
    )

    X_test_cv = add_feature(
        X_test_cv,
        [len_test, dig_test, nan_test]
    )

    clf = LogisticRegression(C=100).fit(X_train_cv, y_train)

    pred = clf.predict(X_test_cv)

    score = roc_auc_score(y_test, pred)

    feature_names = np.concatenate([
        cv.get_feature_names_out(),
        ['length_of_doc', 'digit_count', 'non_word_char_count']
    ])

    sorted_coef_index = clf.coef_[0].argsort()

    small_coeffs = list(
        feature_names[sorted_coef_index[:10]]
    )

    large_coeffs = list(
        feature_names[sorted_coef_index[:-11:-1]]
    )

    return (score, small_coeffs, large_coeffs)

In [70]:
answer_eleven()

(0.9788890209327199,
 ['..', '. ', ' i', 'ok', ' go', 'he', 'i ', '...', 'h ', 'n '],
 ['digit_count', 'ww', 'co', 'xt', 'ne', ' a ', 'uk', 'ar', 'ex', '.co'])